# CW-DETR YOLO BDD100K object-detection training

This Colab notebook trains the CW-DETR Nano detector on a YOLO-format BDD100K export stored in Google Drive.

The provided merged taxonomy has nine object-detection classes. This run is intentionally detection-only: drivable-area masks, lane masks, sign subclassification, learned tracking, and trajectory heads are disabled. Missing YOLO label files are treated as valid background images.

Before running: select `Runtime > Change runtime type > GPU`.

## 1. Mount Google Drive and inspect the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Enable a GPU runtime before training.')

## 2. Clone CW-DETR

While the focused PR is open, the default ref below points at its branch. Change `REPO_REF` to `main` after the PR is merged.

In [ ]:
# @title Repository settings
REPO_URL = 'https://github.com/pirazor/CW-DETR.git' # @param {type:'string'}
REPO_REF = 'codex/yolo-detection-colab' # @param {type:'string'}
REPO_DIR = '/content/CW-DETR'

import os
import subprocess
from pathlib import Path

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_REF], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', REPO_REF, f'origin/{REPO_REF}'], check=True)
os.chdir(REPO_DIR)
revision = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('working directory:', os.getcwd())
print('checked out revision:', revision)

## 3. Install training dependencies

The backbone contract is pinned to the tested DINOv3 dependency line.

In [ ]:
%pip install -q transformers==4.56.0 timm==1.0.11 einops safetensors scipy pycocotools pyyaml tensorboard tqdm pillow

import transformers, timm
print('transformers:', transformers.__version__)
print('timm:', timm.__version__)

## 4. Authenticate for gated DINOv3 weights

Accept the DINOv3 model terms on Hugging Face first. Set `LOGIN_TO_HUGGING_FACE` to `False` only if this runtime already has an authenticated token.

In [ ]:
# @title Hugging Face login
LOGIN_TO_HUGGING_FACE = True # @param {type:'boolean'}

if LOGIN_TO_HUGGING_FACE:
    from huggingface_hub import notebook_login
    notebook_login()

## 5. Validate the YOLO dataset

The adapter reads `train/images/` and `val/images/` from `data.yaml`, then resolves labels from sibling `train/labels/` and `val/labels/` directories. The segmentation fields in your YAML remain available for other trainers but are not consumed by this object-detection-only run.

The first run writes small `.cwdetr-...-images.txt` manifests and parsed `.cwdetr-...-labels.json` caches beside `data.yaml` in Drive. Later sessions reuse them instead of recursively scanning image paths or reopening every YOLO text label. If Ultralytics already created `train/labels.cache` or `val/labels.cache`, CW-DETR imports it. Enable `REBUILD_DATASET_INDEX` only after adding, removing, or editing labels or images.

In [ ]:
# @title Dataset and model config
DATA_YAML = '/content/drive/MyDrive/datasets/bdd100k_merged/data.yaml' # @param {type:'string'}
REBUILD_DATASET_INDEX = False # @param {type:'boolean'}
CONFIG = 'configs/cwdetr_nano_yolo_bdd_detection.yaml'

from cwdetr.config import load_config
import importlib
import cwdetr.data.yolo_detection as yolo_detection
importlib.reload(yolo_detection)
YoloDetectionDataset = yolo_detection.YoloDetectionDataset

EXPECTED_NAMES = [
    'car', 'truck', 'bus', 'train', 'bike', 'cyclist', 'person',
    'traffic_light', 'traffic_sign',
]
cfg = load_config(CONFIG)
train_ds = YoloDetectionDataset(
    DATA_YAML, 'train', expected_num_classes=cfg.model.heads.detection.num_classes,
    refresh_index=REBUILD_DATASET_INDEX)
val_ds = YoloDetectionDataset(
    DATA_YAML, 'val', expected_num_classes=cfg.model.heads.detection.num_classes,
    refresh_index=REBUILD_DATASET_INDEX)
assert train_ds.class_names == EXPECTED_NAMES, (train_ds.class_names, EXPECTED_NAMES)
assert not cfg.model.heads.segmentation.enabled
assert not cfg.model.heads.sign_classification.enabled
print('classes:', train_ds.class_names)
print('train images:', len(train_ds))
print('val images:', len(val_ds))
print('train image dir:', train_ds.image_dir)
print('train label dir:', train_ds.label_dir)
print('cached train manifest:', train_ds.index_path)
print('cached val manifest:', val_ds.index_path)
print('cached parsed train labels:', train_ds.label_cache_path)
print('cached parsed val labels:', val_ds.label_cache_path)

## 6. Visualize one YOLO sample

Adjust `SAMPLE_INDEX` to inspect more labels before starting a long run. Mounted Google Drive occasionally returns a transient `Errno 5` read failure; the loader retries those reads automatically. If Drive remains unavailable after the retries, remount Drive and rerun this cell without rebuilding the dataset index.

In [ ]:
# @title Preview labels
SAMPLE_INDEX = 0 # @param {type:'integer'}

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

sample = train_ds[SAMPLE_INDEX]
image = sample['image']
width, height = image.size
fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(image)
for label, box in zip(sample['labels'].tolist(), sample['boxes'].tolist()):
    cx, cy, bw, bh = box
    x = (cx - bw / 2) * width
    y = (cy - bh / 2) * height
    rect = Rectangle((x, y), bw * width, bh * height, fill=False, linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, train_ds.class_names[label], color='white',
            bbox={'facecolor': 'black', 'alpha': 0.6, 'pad': 2})
ax.set_title(sample['image_id'])
ax.axis('off')
plt.show()

## 7. Configure the run directory and TensorBoard

Run this before training so TensorBoard is open and waiting for event files. It is fine if TensorBoard initially shows no data; scalars appear as soon as the training cell writes them.

In [ ]:
# @title Run directory
OUTPUT_DIR = '/content/drive/MyDrive/cwdetr_runs/bdd100k_yolo_detection' # @param {type:'string'}

from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('run directory:', OUTPUT_DIR)
print('TensorBoard log directory:', f'{OUTPUT_DIR}/tensorboard')

## 8. Start TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $OUTPUT_DIR/tensorboard

## 9. Train the detector

Checkpoints and TensorBoard logs are written to Drive. With a 96 GB GPU you can usually try `BATCH_SIZE = 32` or `64`. If GPU memory is still low and iteration time improves, increase batch size; if data loading or checkpoint writing is the bottleneck, larger batches will not help much. Leave `RESUME` empty for a fresh run.

Recommended defaults: `LEARNING_RATE = 2e-4` for decoder/head/neck parameters, `BACKBONE_LR_MULT = 0.1` so the DINOv3 backbone trains at `2e-5`, AdamW `WEIGHT_DECAY = 1e-4`, cosine decay after `WARMUP_STEPS = 1000`, and validation every epoch. These are standard DETR-style fine-tuning defaults; reduce LR only if loss spikes or becomes NaN.

`WORKERS` controls DataLoader subprocesses. Use `0` for easiest debugging. Try `4` or `8` if Drive is stable and GPU utilization is low. `PREFETCH_FACTOR` controls how many batches each worker prepares ahead of time, and persistent workers avoid respawning them every epoch. Too many workers can still trigger Drive I/O errors, so raise this gradually.

`last_step.pth` can be overwritten every optimizer step inside `OUTPUT_DIR`, while `last_epoch.pth`, numbered epoch checkpoints, and `best_detection_map.pth` are saved after validation. Exact per-step full checkpoints on Google Drive can dominate runtime and leave the GPU idle. For speed, use `STEP_CHECKPOINT_EVERY = 50` or `100`; set it back to `1` only when exact step recovery is more important than throughput. Set `KEEP_STEP_CHECKPOINTS = True` only if you really want thousands of numbered step files.

The launcher streams the child process live and preserves its last 80 log lines if training fails. Startup logs identify whether a failure occurs while loading the model, building caches, creating loaders, training a batch, evaluating, or writing a checkpoint.

In [ ]:
# @title Training settings
EPOCHS = 50 # @param {type:'integer'}
BATCH_SIZE = 32 # @param {type:'integer'}
EVAL_BATCH_SIZE = 4 # @param {type:'integer'}
WORKERS = 4 # @param {type:'integer'}
PREFETCH_FACTOR = 4 # @param {type:'integer'}
PERSISTENT_WORKERS = True # @param {type:'boolean'}
LEARNING_RATE = 0.0002 # @param {type:'number'}
BACKBONE_LR_MULT = 0.1 # @param {type:'number'}
WEIGHT_DECAY = 0.0001 # @param {type:'number'}
CLIP_GRAD_NORM = 0.1 # @param {type:'number'}
WARMUP_STEPS = 1000 # @param {type:'integer'}
EVAL_EVERY = 1 # @param {type:'integer'}
LOG_EVERY = 10 # @param {type:'integer'}
STEP_CHECKPOINT_EVERY = 50 # @param {type:'integer'}
KEEP_STEP_CHECKPOINTS = False # @param {type:'boolean'}
RESUME = '' # @param {type:'string'}

from collections import deque
import os
import shlex
import subprocess
import sys

def run_streaming(command):
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    print('running:', shlex.join(command), flush=True)
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env)
    tail = deque(maxlen=80)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        tail.append(line)
    returncode = process.wait()
    if returncode:
        recent = ''.join(tail)
        hint = ''
        if 'CUDA out of memory' in recent:
            hint = '\n\nHint: reduce BATCH_SIZE to 2 or 4 and rerun the training cell.'
        raise RuntimeError(
            f'Command failed with exit code {returncode}. Last child-process logs:\n\n{recent}{hint}')

cmd = [
    sys.executable, '-u', '-m', 'cwdetr.engine.train',
    '--config', CONFIG,
    '--yolo-data', DATA_YAML,
    '--out', OUTPUT_DIR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--eval-batch-size', str(EVAL_BATCH_SIZE),
    '--workers', str(WORKERS),
    '--prefetch-factor', str(PREFETCH_FACTOR),
    '--lr', str(LEARNING_RATE),
    '--backbone-lr-mult', str(BACKBONE_LR_MULT),
    '--weight-decay', str(WEIGHT_DECAY),
    '--clip-grad-norm', str(CLIP_GRAD_NORM),
    '--warmup-steps', str(WARMUP_STEPS),
    '--eval-every', str(EVAL_EVERY),
    '--log-every', str(LOG_EVERY),
    '--step-checkpoint-every', str(STEP_CHECKPOINT_EVERY),
]
if not PERSISTENT_WORKERS:
    cmd += ['--no-persistent-workers']
if KEEP_STEP_CHECKPOINTS:
    cmd += ['--keep-step-checkpoints']
if RESUME.strip():
    cmd += ['--resume', RESUME]
run_streaming(cmd)

## 10. Evaluate the best checkpoint

For this detection-only config, `detection/map` and `detection/map50` are the relevant metrics. Segmentation and sign metrics remain zero by design.

In [ ]:
# @title COCO-style validation
BEST_CHECKPOINT = f'{OUTPUT_DIR}/best_detection_map.pth'
eval_cmd = [
    sys.executable, '-u', '-m', 'cwdetr.engine.evaluate',
    '--config', CONFIG,
    '--yolo-data', DATA_YAML,
    '--ckpt', BEST_CHECKPOINT,
    '--batch-size', str(EVAL_BATCH_SIZE),
    '--workers', str(WORKERS),
]
run_streaming(eval_cmd)